#### 1. Load libraries, functions, config file...

In [ ]:
# Update files
import importlib
import config as cfg
import utils.io
import utils.metrics
import data.data_loader
import data.data_augmentation
import models.multi_layer_perceptron
import models.convolutional_neural_networks
import models.residual_network

importlib.reload(cfg)
importlib.reload(utils.io)
importlib.reload(utils.metrics)
importlib.reload(data.data_loader)
importlib.reload(data.data_augmentation)
importlib.reload(models.multi_layer_perceptron)
importlib.reload(models.convolutional_neural_networks)
importlib.reload(models.residual_network)

# Import libraries  
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split

# Import functions
## Data
from data.data_loader import ButterflyDataset
from data.data_augmentation import apply_data_augmentation
from utils.io import print_images
from utils.metrics import analyze_df


## Utils
from utils.io import save_acc_graph
## Metric
from utils.metrics import evaluate_network

## MLP
from models.multi_layer_perceptron import MLP, fit as mlp_fit

## CNN
from models.convolutional_neural_networks import CNN, fit as cnn_fit

## ResNet
from models.residual_network import ResNet, fit as rnet_fit

# Choose device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():    
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

#### 2. Load / Augment DATA

In [ ]:
# load the data
df = pd.read_csv(cfg.TRAIN_LABELS_PATH)

df_train, df_val = train_test_split(df, test_size=cfg.TEST_SIZE, random_state=42, stratify=df['label'])

workers = min(4, os.cpu_count() or 1)

if cfg.AUGMENT_DATA:
    train_transform, val_transform, sampler = apply_data_augmentation(df_train)

    # preprocessing (Datasets utilizam transforms independentes)
    train_dataset = ButterflyDataset(df=df_train, img_dir=cfg.TRAIN_IMG_DIR, transform=train_transform)
    val_dataset = ButterflyDataset(df=df_val, img_dir=cfg.TRAIN_IMG_DIR, transform=val_transform)
    
    train_loader = data.DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, sampler=sampler, num_workers=workers)
    val_loader = data.DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=workers)
    print("All images processed with SOTA Data Augmentation and Class Balancing...")
else:
    # preprocessing (Não aplica-se data augmentation)
    data_transform = transforms.Compose([
        transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
        transforms.ToTensor()
    ])
    
    train_dataset = ButterflyDataset(df=df_train, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)
    val_dataset = ButterflyDataset(df=df_val, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)

    train_loader = data.DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=workers)
    val_loader = data.DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=workers)
    print("All images processed with SOTA Data Augmentation without Class Balancing...")


#### 3. Analyze Dataset

In [ ]:
# Train data set analyze
num_inputs, num_classes = analyze_df(df_train, train_dataset)
print_images(train_dataset, train_loader)

#### 4. MLP

In [ ]:
dnn = MLP(input_size=num_inputs, hidden_sizes=cfg.HIDDEN_LAYER_SIZES, num_classes=num_classes)

# Escolher Loss Function
if cfg.LOSS_FUNCTION == "CrossEntropy":
    criterion = nn.CrossEntropyLoss()
elif cfg.LOSS_FUNCTION == "CrossEntropy_LabelSmoothing":
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING) 
elif cfg.LOSS_FUNCTION == "MultiMarginLoss":
    criterion = nn.MultiMarginLoss()
else:
    raise ValueError(f"Loss Function desconhecida no config: {cfg.LOSS_FUNCTION}")

# Escolher Otimizador
if cfg.OPTIM == "RMSprop":
    optimizer = optim.RMSprop(dnn.parameters(), lr=cfg.LR)
elif cfg.OPTIM == "ADAM":
    optimizer = optim.Adam(dnn.parameters(), lr=cfg.LR)
elif cfg.OPTIM == "ADAMW":
    optimizer = optim.AdamW(dnn.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
elif cfg.OPTIM == "SGD":
    optimizer = optim.SGD(dnn.parameters(), lr=cfg.LR, momentum=cfg.MOMENTUM)
else:
    raise ValueError(f"Otimizador desconhecido no config: {cfg.OPTIM}")

# Treino
print(f"A iniciar o treino do MLP com {cfg.OPTIM} e {cfg.LOSS_FUNCTION}...")
mlp_train_acc, mlp_val_acc, dnn = mlp_fit(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    nn=dnn,
    criterion=criterion, 
    optimizer=optimizer, 
    n_epochs=cfg.N_EPOCHS, 
    to_device=False,
    device=device
)


#### 4.1. Evaluate MLP

In [ ]:
if cfg.AUGMENT_DATA:
    mlp_result_file_path = cfg.mlp_results_path_augmented
else:
    mlp_result_file_path = cfg.mlp_results_path

print('Saving graphics showing acc results...')
save_acc_graph(mlp_train_acc, mlp_val_acc, mlp_result_file_path, cfg.mlp_results_folder_name, "RNN")

print('Evaluating with the training data...')
evaluate_network(
    net=dnn, 
    dataloader=train_loader, 
    device=device,
    save_path=mlp_result_file_path, 
    split_name="Treino"
)

print('Evaluating with the valid data...')
evaluate_network(
    net=dnn, 
    dataloader=val_loader, 
    device=device, 
    save_path=mlp_result_file_path,
    split_name="Validacao"
)

#### 5. CNN

In [ ]:
cnn = CNN(input_channels=3, num_classes=num_classes)

# Escolher Loss Function
if cfg.LOSS_FUNCTION == "CrossEntropy":
    criterion = nn.CrossEntropyLoss()
elif cfg.LOSS_FUNCTION == "CrossEntropy_LabelSmoothing":
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING) 
elif cfg.LOSS_FUNCTION == "MultiMarginLoss":
    criterion = nn.MultiMarginLoss()
else:
    raise ValueError(f"Loss Function desconhecida no config: {cfg.LOSS_FUNCTION}")

# Escolher Otimizador
if cfg.OPTIM == "RMSprop":
    optimizer = optim.RMSprop(cnn.parameters(), lr=cfg.LR)
elif cfg.OPTIM == "ADAM":
    optimizer = optim.Adam(cnn.parameters(), lr=cfg.LR)
elif cfg.OPTIM == "ADAMW":
    optimizer = optim.AdamW(cnn.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
elif cfg.OPTIM == "SGD":
    optimizer = optim.SGD(cnn.parameters(), lr=cfg.LR, momentum=cfg.MOMENTUM)
else:
    raise ValueError(f"Otimizador desconhecido no config: {cfg.OPTIM}")

# Treino
print(f"\nA iniciar o treino da CNN com {cfg.OPTIM} e {cfg.LOSS_FUNCTION}...")
cnn_train_acc, cnn_val_acc, cnn = cnn_fit(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    nn=cnn,
    criterion=criterion, 
    optimizer=optimizer, 
    n_epochs=cfg.N_EPOCHS, 
    to_device=False,
    device=device
)

#### 5.1. Evaluate CNN

In [ ]:
if cfg.AUGMENT_DATA:
    cnn_result_file_path = cfg.cnn_results_path_augmented
else:
    cnn_result_file_path = cfg.cnn_results_path

print('Saving graphics showing acc results...')
save_acc_graph(cnn_train_acc, cnn_val_acc, cnn_result_file_path, cfg.cnn_results_folder_name, "CNN")

print('Evaluating with the training data...')
evaluate_network(
    net=cnn, 
    dataloader=train_loader, 
    device=device,
    save_path=cnn_result_file_path, 
    split_name="Treino",
)

print('Evaluating with the valid data...')
evaluate_network(
    net=cnn, 
    dataloader=val_loader, 
    device=device, 
    save_path=cnn_result_file_path, 
    split_name="Validacao",
)

#### 6. ResNet

In [ ]:
rnet = ResNet(num_classes=num_classes)

# Escolher Loss Function
if cfg.LOSS_FUNCTION == "CrossEntropy":
    criterion = nn.CrossEntropyLoss()
elif cfg.LOSS_FUNCTION == "CrossEntropy_LabelSmoothing":
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING) 
elif cfg.LOSS_FUNCTION == "MultiMarginLoss":
    criterion = nn.MultiMarginLoss()
else:
    raise ValueError(f"Loss Function desconhecida no config: {cfg.LOSS_FUNCTION}")

# Escolher Otimizador
if cfg.OPTIM == "RMSprop":
    optimizer = optim.RMSprop(rnet.parameters(), lr=cfg.LR)
elif cfg.OPTIM == "ADAM":
    optimizer = optim.Adam(rnet.parameters(), lr=cfg.LR)
elif cfg.OPTIM == "ADAMW":
    optimizer = optim.AdamW(rnet.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
elif cfg.OPTIM == "SGD":
    optimizer = optim.SGD(rnet.parameters(), lr=cfg.LR, momentum=cfg.MOMENTUM)
else:
    raise ValueError(f"Otimizador desconhecido no config: {cfg.OPTIM}")

# Treino
print(f"\nA iniciar o treino da ResNet com {cfg.OPTIM} e {cfg.LOSS_FUNCTION}...")
rnet_train_acc, rnet_val_acc, rnet = rnet_fit(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    nn=rnet,
    criterion=criterion, 
    optimizer=optimizer, 
    n_epochs=cfg.N_EPOCHS, 
    to_device=False,
    device=device
)

#### 6.1. Evaluate ResNet

In [ ]:
if cfg.AUGMENT_DATA:
    rnet_result_file_path = cfg.rnet_results_path_augmented
else:
    rnet_result_file_path = cfg.rnet_results_path

print('Saving graphics showing acc results...')
save_acc_graph(rnet_train_acc, rnet_val_acc, rnet_result_file_path, cfg.rnet_results_folder_name, "RNET")

print('Evaluating with the training data...')
evaluate_network(
    net=rnet, 
    dataloader=train_loader, 
    device=device,
    save_path=rnet_result_file_path, 
    split_name="Treino",
)

print('Evaluating with the valid data...')
evaluate_network(
    net=rnet, 
    dataloader=val_loader, 
    device=device, 
    save_path=rnet_result_file_path, 
    split_name="Validacao",
)